# Early Breast Cancer Diagnostic Support Using Machine Learning

### Predicting Benign and Malignant Tumors with Interpretable Machine Learning

**Author:** Alyssa Seal

**Tools:** Python • Pandas • Scikit-learn • XGBoost • SHAP • Matplotlib

**Objective**

Develop and evaluate machine learning models capable of predicting breast cancer diagnoses using diagnostic tumor measurements while emphasizing model interpretability and clinical decision support.

---

# Data Acquisition

The enhanced breast cancer diagnostic dataset was downloaded directly from Kaggle using the KaggleHub API. The dataset contains diagnostic measurements describing tumor morphology together with a binary diagnosis indicating whether each tumor is benign or malignant.

In [ ]:
import kagglehub

# Download latest version of dataset from kaggle 
path = kagglehub.dataset_download("shivasingh4945/enhanced-breast-cancer-diagnostic-dataset")

print("Path to dataset files:", path)

In [ ]:
import os 
for file in os.listdir(path):
    print(file)

In [ ]:
# Set up reusable save path for figures 

FIGURES_DIR = "figures"

os.makedirs(FIGURES_DIR, exist_ok=True)

In [ ]:
# Load the dataset 
import pandas as pd

csv_file = os.path.join(path, "breast_cancer_enhanced_dataset.csv")

df = pd.read_csv(csv_file)

df.head()

---

# Initial Data Exploration

Before model development, the dataset was examined to understand its structure, variable types, overall size, and potential data quality issues.

In [ ]:
# Explore the dataset 

print(df.shape)

df.info()

In [ ]:
df.describe()
df.isnull().sum()

In [ ]:
df.columns

In [ ]:
df['diagnosis'].value_counts()

---

# Exploratory Data Analysis

Exploratory analysis was performed to understand the distribution of diagnoses, identify relationships among variables, and detect potential issues such as missing values or highly correlated features.

In [ ]:
# Distribution map to visualize diagnoses
import matplotlib.pyplot as plt

df['diagnosis'].value_counts().plot(kind='bar')

plt.title('Breast Cancer Diagnosis Distribution')
plt.xlabel('Diagnosis')
plt.ylabel('Count')

plt.savefig(
    "figures/breast_cancer_diagnosis_dsitribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Correlation heatmap 

import seaborn as sns

plt.figure(figsize=(12, 10))

sns.heatmap(
    df.corr(numeric_only=True), 
    cmap='coolwarm', 
    center=0
)

plt.title('Feature Correlation Heatmap')

plt.savefig(
    "figures/feature_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## Target Variable

The diagnosis variable serves as the prediction target.

- **0 = Benign**
- **1 = Malignant**

Encoding the diagnosis as a binary variable allows the dataset to be used for supervised classification.

In [ ]:
# Target variable is 'diagnosis' with values B= Benign (non-cancerous) and M= Malignant

df['diagnosis'] = df['diagnosis'].map({
    'B': 0, 
    'M': 1
})

In [ ]:
# Investigate engineered features from data set that would potentially skew results:
# tumor_aggressiveness and malignancy_risk_score

df.groupby('diagnosis')['malignancy_risk_score'].describe()


In [ ]:
df.groupby('diagnosis')['tumor_aggressiveness'].describe()

---

# Data Cleaning and Feature Selection

Several variables were removed prior to model training to prevent data leakage.

- `id` was removed because it provides no predictive information.
- `tumor_aggressiveness` was removed because it likely incorporates post-diagnostic information.
- `malignancy_risk_score` was removed because it appears to directly summarize malignancy risk and would artificially inflate model performance.

In [ ]:
# Drop unnecessary columns (and columns that were engineered most likely post diagnosis)

df = df.drop('id', axis=1)
df = df.drop('malignancy_risk_score', axis = 1)
df = df.drop('tumor_aggressiveness', axis =1)

In [ ]:
# Check balance 

df['diagnosis'].value_counts()
df['diagnosis'].value_counts(normalize=True) * 100

In [ ]:
# Check for correlation to determine what features are associated with malignancy

corr = df.corr(numeric_only=True)

corr['diagnosis'].sort_values(ascending=False)

## Train-Test Split

The dataset was divided into training and testing subsets using an 80/20 split with stratified sampling to preserve the class distribution in both datasets.

In [ ]:
# Split the data into training and testing sets 

from sklearn.model_selection import train_test_split

X= df.drop('diagnosis', axis = 1)
y = df['diagnosis']

X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size = 0.2, 
    stratify=y, 
    random_state = 42
)

---

# Baseline Model

A Dummy Classifier was used to establish a performance benchmark.

Any predictive model should outperform this baseline before being considered useful.

In [ ]:
# Baseline model to establish benchmark

from sklearn.dummy import DummyClassifier 

dummy= DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

In [ ]:
# Test dummy model 

from sklearn.metrics import accuracy_score

dummy_pred = dummy.predict(X_test)

print("Baseline Accuracy:", accuracy_score(y_test, dummy_pred))

---

# Logistic Regression

Logistic Regression serves as an interpretable baseline model that is commonly used in healthcare because coefficients can be directly related to changes in predicted risk.

In [ ]:
# Logistic Regression Model 

from sklearn.pipeline import Pipeline 
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression 

log_model = Pipeline([
    ('scaler', StandardScaler()), 
    ('model', LogisticRegression(max_iter = 1000))
])

log_model.fit(X_train, y_train)

In [ ]:
log_pred = log_model.predict(X_test)
log_prob = log_model.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix, 
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, log_pred))

print("\nClassification Report:")
print(classification_report(y_test, log_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_test, log_prob))

In [ ]:
# Confusion Matrix Visualization 

from sklearn.metrics import ConfusionMatrixDisplay 

ConfusionMatrixDisplay.from_predictions(
    y_test, 
    log_pred
)

plt.title("Logistic Regression Confusion Matrix")

plt.savefig(
    "figures/logistic_regression_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ROC Curve 

from sklearn.metrics import RocCurveDisplay 

RocCurveDisplay.from_predictions(
    y_test, 
    log_prob
)

plt.title("Logistic Regression ROC Curve")

plt.savefig(
    "figures/logistic_regression_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
coefficients = pd.DataFrame({
    'Feature': X.columns, 
    'Coefficient': log_model.named_steps['model'].coef_[0]
})

coefficients = coefficients.sort_values(
    by='Coefficient', 
    ascending=False
)

print(coefficients)

---

# Random Forest

Random Forest is an ensemble learning algorithm capable of capturing complex nonlinear relationships while reducing overfitting through the use of multiple decision trees.

In [ ]:
# Random Forest Classifier 

from sklearn.ensemble import RandomForestClassifier 

rf= RandomForestClassifier(
    n_estimators=300, 
    random_state = 42, 
    n_jobs = -1
)

rf.fit(X_train, y_train)

In [ ]:
rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, rf_pred))

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_test, rf_prob))

In [ ]:
# Confussion Matrix for randomforest 

from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt 

ConfusionMatrixDisplay.from_predictions(
    y_test, 
    rf_pred
)

plt.title("Random Forest Confusion Matrix")

plt.savefig(
    "figures/random_forest_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# ROC curve for randomforest 

from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_predictions(
    y_test, 
    rf_prob
)

plt.title("Random Forest ROC Curve")

plt.savefig(
    "figures/random_forest_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Feature importance 

feature_importance = pd.DataFrame({
    'Feature': X.columns, 
    'Importance': rf.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance', 
    ascending=False
)

print(feature_importance)

In [ ]:
# Visualize top features

top_features = feature_importance.head(10)

plt.figure(figsize=(10,6))

plt.barh(
    top_features['Feature'], 
    top_features['Importance']
)

plt.title("Top 10 Random Forest Feature Importances")
plt.xlabel("Importance")

plt.tight_layout()

plt.savefig(
    "figures/top_10_random_forest_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Cross-validation 

from sklearn.model_selection import cross_val_score

rf_cv = cross_val_score(
    rf, 
    X_train,
    y_train, 
    cv=5, 
    scoring='roc_auc'
)

print("CV ROC-AUC Scores:", rf_cv)
print("Mean ROC-AUC:", rf_cv.mean())

---

# Extreme Gradient Boosting (XGBoost)

XGBoost is an advanced gradient boosting algorithm known for its excellent performance on structured tabular datasets. It was included to evaluate whether boosting methods outperform traditional ensemble approaches.

In [ ]:
# XG Boost Model 

from xgboost import XGBClassifier 

xgb = XGBClassifier(
    n_estimators = 300, 
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=42, 
    eval_metric = 'logloss'
)

xgb.fit(X_train, y_train)

In [ ]:
# Generate predictions 

xgb_pred = xgb.predict(X_test)
xgb_prob = xgb.predict_proba(X_test)[:,1]

In [ ]:
# Evaluate XG Boost Model 

print("Accuracy:", accuracy_score(y_test, xgb_pred))

print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))

print("\nROC-AUC:")
print(roc_auc_score(y_test, xgb_prob))

In [ ]:
# Confusion Matrix for XG Boost 

ConfusionMatrixDisplay.from_predictions(
    y_test, 
    xgb_pred
)

plt.title("XGBoost Confusion Matrix")

plt.savefig(
    "figures/xgboost_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# XG Boost ROC Curve 

RocCurveDisplay.from_predictions(
    y_test, 
    xgb_prob
)

plt.title("XGBoost ROC Curve")

plt.savefig(
    "figures/xgboost_roc_curve.png", 
    dpi=300, 
    bbox_inches="tight"
)

plt.show()

In [ ]:
xgb_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb.feature_importances_
})

xgb_importance = xgb_importance.sort_values(
    by="Importance",
    ascending=False
)

print(xgb_importance)

In [ ]:
top_features = xgb_importance.head(10)

plt.figure(figsize=(10,6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 10 XGBoost Feature Importances")
plt.xlabel("Importance")

plt.tight_layout()

plt.savefig(
    "figures/xgboost_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

---

# Model Comparison

The performance of all models was compared using multiple evaluation metrics.

Because this project focuses on cancer detection, recall and ROC-AUC were considered particularly important, as correctly identifying malignant tumors is generally more critical than minimizing false positives.

In [ ]:
# Summary table prep 

from sklearn.metrics import accuracy_score, roc_auc_score

dummy_pred = dummy.predict(X_test)

dummy_accuracy = accuracy_score(y_test, dummy_pred)

# Dummy doesn't produce probabilities by default
dummy_auc = 0.5

# Logistic Regression 
log_accuracy = accuracy_score(y_test, log_pred)

log_auc = roc_auc_score(y_test, log_prob)

# Random Forest 
rf_accuracy = accuracy_score(y_test, rf_pred)

rf_auc = roc_auc_score(y_test, rf_prob)

# XGBoost 
xgb_accuracy = accuracy_score(y_test, xgb_pred)

xgb_auc = roc_auc_score(y_test, xgb_prob)


In [ ]:
# Summary table 

results = pd.DataFrame({
    "Model": [
        "Dummy",
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        dummy_accuracy,
        log_accuracy,
        rf_accuracy,
        xgb_accuracy
    ],
    "ROC_AUC": [
        dummy_auc,
        log_auc,
        rf_auc,
        xgb_auc
    ]
})

results.sort_values(
    by="ROC_AUC",
    ascending=False
)

---

# Example Predictions

Several observations from the test set were evaluated individually to demonstrate how the trained XGBoost model predicts benign and malignant tumors in practice.

In [ ]:
# Testing 

for i in range(5):
    
    sample = X_test.iloc[[i]]

    pred = xgb.predict(sample)[0]

    prob = xgb.predict_proba(sample)[0][1]

    actual = y_test.iloc[i]

    print(f"\nCase {i+1}")
    print("Predicted:", "Malignant" if pred == 1 else "Benign")
    print("Actual:", "Malignant" if actual == 1 else "Benign")
    print(f"Probability Malignant: {prob:.2%}")

---

# Explainable Artificial Intelligence with SHAP

Machine learning models can achieve high predictive performance while remaining difficult to interpret.

SHAP (SHapley Additive exPlanations) was used to quantify how individual tumor characteristics influenced predictions, providing greater transparency into the model's decision-making process.

In [ ]:
!pip install shap

In [ ]:
import shap 

explainer = shap.TreeExplainer(xgb)

shap_values = explainer.shap_values(X_test)

In [ ]:
# Global feature importance plot
shap.summary_plot(
    shap_values, 
    X_test, 
    plot_type='bar', 
    show=False
)

plt.title("SHAP Feature Importance - XGBoost")
plt.tight_layout()

plt.savefig(
    "figures/shap_feature_importance.png", 
    dpi=300, 
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Detailed SHAP summary plot 

shap.summary_plot(
    shap_values, 
    X_test, 
    show=False
)

plt.savefig(
    "figures/shap_summary_plot.png", 
    dpi=300, 
    bbox_inches="tight"
)

plt.show()

In [ ]:
sample_index = 0 

sample = X_test.iloc[[sample_index]]

prediction = xgb.predict(sample)[0]
probability = xgb.predict_proba(sample)[0][1]
actual = y_test.iloc[sample_index]

print("Predicted:", "Malignant" if prediction == 1 else "Benign")
print("Actual:", "Malignant" if prediction ==1 else "Benign")
print(f"Probability Malignant: {probability:.2%}")

In [ ]:
shap.initjs()

shap.force_plot(
    explainer.expected_value, 
    shap_values[sample_index], 
    X_test.iloc[sample_index]
)

In [ ]:
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_values[sample_index],
    X_test.iloc[sample_index],
    show=False
)

plt.savefig(
    "figures/shap_waterfall_example.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

---

# Conclusions

Three machine learning algorithms were evaluated for early breast cancer classification.

Among the evaluated models, XGBoost achieved the strongest predictive performance while SHAP analysis identified the tumor characteristics contributing most strongly to malignant predictions.

This project demonstrates a complete healthcare machine learning workflow including:

- Data acquisition
- Exploratory data analysis
- Data cleaning and feature selection
- Predictive modeling
- Model evaluation
- Explainable AI using SHAP

The resulting workflow illustrates how interpretable machine learning can support clinical decision-making while maintaining transparency regarding the factors influencing model predictions.